# 📝 정보 추출·NER 과제 LV1(기초): 개체 스키마·검증·집계

> 이 단원의 새 기술을 **하나씩** 확인합니다. 문제는 다섯 갈래로 묶여 있습니다.
>
> - **1. 구간과 BIO 태깅**: `find` 로 구간(`start`·`end`) 만들기 · 토큰마다 `B-`/`I-`/`O` 붙이기 · 태그에서 다시 구간 되살리기
> - **2. 개체 스키마 정의**: pydantic `BaseModel`·`Field(description=...)` · 개체 하나와 개체 목록
> - **3. 뽑은 개체 검증**: 허용 유형만 남기기와 그 반대 · 원문 등장 확인과 그 반대(환각 골라내기)
> - **4. 정리와 집계**: `(이름, 유형)` 중복 제거 · `Counter` 로 유형별 개수 세기
> - **5. LLM 으로 뽑기**: `with_structured_output` 추출기 만들기 · 규칙 기반의 한계(서술형)

## 풀이 방법
1. 맨 위 **준비 셀**을 먼저 실행하세요(`.env` 에 `OPENAI_API_KEY` 가 있어야 합니다).
2. 각 문제의 **답안 셀**에 코드를 채우고 **자가채점 셀**로 확인하세요(✅ 통과!).

- 데이터: PMC 논문 발췌 6편(신장·대사와 그 약물)입니다. 고정 추출 파일 `data/lv1_entities.jsonl` 과 원문 `data/lv1_papers.jsonl` 을 씁니다.
- 개체 유형은 교안과 같은 5종입니다. `Compound`·`Gene`·`Disease`·`Symptom`·`PharmacologicClass`.

화이팅!

아래 준비 셀을 먼저 실행하세요.

In [ ]:
# [제공 코드] OpenAI 키 준비 - 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

from langchain_openai import ChatOpenAI

MODEL_NAME = "gpt-5.6-luna"


def make_model():
    """이 단원이 쓰는 챗 모델.

    이 모델은 temperature 를 받지 않습니다(0 을 넘기면 400 이 옵니다). 그래서 출력을 고정할
    손잡이가 없고, 같은 입력에도 답이 흔들립니다(자동화 파이프라인 단원 6절이 그 흔들림을
    여러 번 돌려 잽니다).
    """
    return ChatOpenAI(model=MODEL_NAME)


print("모델:", MODEL_NAME)

In [ ]:
# [제공 코드] 데이터 파일 읽기: 이 셀은 실행만 하세요.
import json
from pathlib import Path

# 교안은 단원 폴더에서, 정답 노트북은 정답/ 폴더에서 돌아가므로 두 경로를 모두 본다.
_DATA = Path("data") if Path("data").exists() else Path("../data")


def load_jsonl(name):
    """data/<name> 을 한 줄씩 읽어 dict 리스트로 돌려준다(한 줄에 JSON 하나)."""
    rows = []
    for line in (_DATA / name).read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


def load_json(name):
    """data/<name> 을 통째로 읽어 dict 로 돌려준다(사전 파일용)."""
    return json.loads((_DATA / name).read_text(encoding="utf-8"))


print("데이터 폴더:", _DATA)

In [ ]:
# [제공 코드] 스키마 도구 준비: 2-1, 2-2, 5-1 에서 쓸 도구를 미리 가져옵니다(실행만 하세요).
from typing import Literal, Optional
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

print('스키마 도구 준비 완료: BaseModel, Field, Literal, ChatPromptTemplate')

## 데이터 살펴보기
아래 셀은 **실행만** 하면 됩니다. 문제의 재료가 되는 개체 추출 결과와 논문 원문을 먼저 훑어봅니다. 추출 결과에는 일부러 **결함**(허용 안 된 유형·원문에 없는 개체·중복)이 섞여 있습니다.

In [ ]:
# [제공 코드] 이 셀은 실행만 하세요. 문제의 재료가 되는 개체와 원문을 읽어 둡니다.
lv1_entities = load_jsonl('lv1_entities.jsonl')   # 추출된 개체(결함 포함)
lv1_papers = load_jsonl('lv1_papers.jsonl')       # 논문 발췌 원문
doc_text = {paper['doc_id']: paper['text'] for paper in lv1_papers}
allowed_types = {'Compound', 'Gene', 'Disease', 'Symptom', 'PharmacologicClass'}   # 그래프 노드 레이블과 같은 5종

print('개체 수:', len(lv1_entities), '/ 논문 수:', len(lv1_papers))
for r in lv1_entities[:5]:
    print(r)

---
# 1. 구간과 BIO 태깅

NER 의 출력 형식을 손으로 만들어 봅니다. `find` 로 구간을 잡고, 그 구간을 토큰마다의 라벨로 옮겨 적습니다(교안_01 2-1~2-2).

## 1-1. 개체의 구간(span) 만들기
**배경**: NER 의 출력은 **(구간, 유형)** 쌍입니다. 구간은 "원문에서 몇 번째 글자부터 몇 번째 글자까지"인지를 가리킵니다. 아래 제공된 문장에서 개체 세 개의 구간을 직접 만들어 봅니다.

**요구사항**:
- 제공된 `labeled` 의 `(이름, 유형)` 을 돌며 `sentence.find(이름)` 로 시작 위치를 구하세요.
- 개체마다 **`{'name', 'type', 'start', 'end'}`** 네 키를 가진 dict 를 만들어 **`spans`** 리스트에 담으세요(`end` 는 시작 위치에 이름 길이를 더한 값).
- 각 구간으로 원문을 되짚어(`sentence[start:end]`) 이름과 같은지 눈으로 확인하며 출력하세요.

**예시**: `spans[0]` 은 `{'name': 'adenine', 'type': 'Compound', 'start': 29, 'end': 36}` 이고, `sentence[29:36]` 은 `'adenine'` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- find 로 시작 위치를 얻고, 거기에 이름 길이를 더해 끝 위치를 만든다.

세부구현:
1. 빈 리스트 spans 를 만들고 labeled 를 돈다.
2. sentence.find(이름) 으로 start 를 구한다(원문에 없으면 -1 이 돌아온다).
3. end 는 start + len(이름) 이다. end 는 마지막 글자의 다음 자리라 슬라이싱에 그대로 쓴다.
4. 네 키를 담은 dict 를 spans 에 넣고, sentence[start:end] 와 함께 출력한다.
```

</details>

In [ ]:
# [제공 코드] 1-1, 1-2 가 쓸 문장과 사람이 눈으로 찾아 둔 개체입니다(실행만 하세요).
sentence = 'ATP is then metabolized into adenine and ultimately converted into uric acid, which elevates serum uric acid levels and increases the risk of gout.'
labeled = [('adenine', 'Compound'), ('uric acid', 'Compound'), ('gout', 'Disease')]
print(sentence)

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(spans) == len(labeled), 'labeled 의 개체 수만큼 담아야 합니다'
assert all(set(s) == {'name', 'type', 'start', 'end'} for s in spans), '키는 name, type, start, end 네 개입니다'
assert all(s['start'] != -1 for s in spans), 'start 가 -1 이면 원문에서 이름을 못 찾은 것입니다'
assert [(s['name'], s['type']) for s in spans] == labeled, 'labeled 의 순서와 이름, 유형을 그대로 옮기세요'
assert all(sentence[s['start']:s['end']] == s['name'] for s in spans), \
    '구간으로 원문을 되짚었을 때 개체 이름과 같아야 합니다(end 는 start + len(name) 입니다)'
print('✅ 통과!')

## 1-2. BIO 태깅으로 적기
**배경**: 구간을 숫자 쌍으로 저장할 수도 있지만, NER 모델을 **학습**시킬 때는 토큰마다 라벨을 하나씩 붙인 **BIO 태깅**으로 적습니다. `B-유형`은 개체가 시작하는 토큰, `I-유형`은 그 개체가 이어지는 토큰, `O` 는 개체가 아닌 토큰입니다.

**요구사항**:
- `sentence.split()` 으로 토큰 리스트 **`tokens`** 를 만드세요.
- 전부 `'O'` 로 채운 리스트 **`tags`** 를 만든 뒤, `labeled` 의 개체에 걸린 토큰만 덮어쓰세요.
- 개체 이름을 `split()` 하면 낱말 몇 개짜리인지 알 수 있습니다. 첫 낱말이 **들어 있는**(`in`) 토큰을 찾아, 그 자리부터 첫 토큰에 `'B-유형'`, 이어지는 토큰에 `'I-유형'` 을 붙입니다.
- 같은 이름이 뒤에 또 나와도 **첫 자리만** 태깅하고 넘어갑니다(수업용 단순화).
- 토큰과 라벨을 짝지어 한 줄씩 출력하세요.

**예시**: 토큰은 24개이고 `tags` 에는 `'B-Compound'` 가 두 번, `'I-Compound'` 와 `'B-Disease'` 가 한 번씩 들어갑니다.

<details><summary>힌트</summary>

```text
접근방법:
- 먼저 전부 O 로 두고, 개체에 걸린 자리만 덮어쓴다.

세부구현:
1. tokens 를 만들고 tags 를 ['O'] * len(tokens) 로 초기화한다.
2. labeled 를 돌며 이름을 split() 해 낱말 목록 parts 를 만든다.
3. enumerate(tokens) 로 돌면서 parts[0] 이 토큰 안에 있는지(in) 검사한다.
   3-1. 완전일치가 아니라 in 을 쓰는 이유는 쉼표·마침표가 토큰에 붙어 있기 때문이다.
4. 찾으면 그 자리부터 parts 길이만큼 B-/I- 를 붙이고 break 로 빠져나온다.
5. zip(tokens, tags) 로 짝지어 출력한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert tokens == sentence.split(), 'tokens 는 sentence.split() 결과입니다'
assert len(tags) == len(tokens), '토큰 하나에 라벨 하나입니다'
assert tags.count('B-Compound') == 2 and tags.count('I-Compound') == 1, \
    "B-/I- 개수가 다릅니다. 'uric acid' 는 B-Compound 다음에 I-Compound 가 와야 합니다"
assert tags.count('B-Disease') == 1 and tags.count('O') == len(tokens) - 4, \
    'gout 에 B-Disease 를 붙이고, 개체가 아닌 토큰은 모두 O 로 두세요'
_i = tags.index('I-Compound')
assert tags[_i - 1] == 'B-Compound' and tokens[_i - 1] == 'uric', \
    "I- 는 같은 개체의 B- 바로 뒤에 이어져야 합니다('uric' 다음이 'acid,')"
assert tags[tokens.index('adenine')] == 'B-Compound' and tags[tokens.index('gout.')] == 'B-Disease', \
    "adenine 과 gout 에 각각 B-Compound, B-Disease 가 붙었는지 확인하세요"
print('✅ 통과!')

In [ ]:
# [제공 코드] 1-3 이 쓸 NER 모델 도구와 문장입니다(실행만 하세요).
# 처음 실행하면 모델을 약 430MB 내려받습니다. 그 뒤로는 캐시에서 바로 뜹니다.
from transformers import logging as hf_logging, pipeline

hf_logging.set_verbosity_error()     # 모델을 올릴 때 뜨는 안내 표를 끈다
hf_logging.disable_progress_bar()    # 가중치를 올리는 진행 막대를 끈다

news_sentence = 'Daniel Park joined Elastic in Amsterdam last March.'
print(news_sentence)

## 1-3. 학습된 NER 모델의 출력 읽기
**배경**: 1-1·1-2 는 사람이 손으로 붙인 라벨이었습니다. **학습된 NER 모델**은 그 라벨을 어떤 모양으로 내놓을까요. 교안_01 4절에서 쓴 그 모델을 다른 문장에 돌려, 출력을 읽는 연습을 합니다.

**요구사항**:
- `pipeline('token-classification', model='dslim/bert-base-NER')` 로 만든 파이프라인을 **`ner`** 에 담고, 제공된 **`news_sentence`** 에 적용한 결과를 **`raw_tags`** 에 담으세요(리스트).
- `raw_tags` 의 각 원소에서 **`entity`** 값만 순서대로 모아 **`bio_labels`** 리스트에 담으세요.
- 같은 모델을 **`aggregation_strategy='simple'`** 로 한 번 더 만들어 같은 문장에 적용한 결과를 **`merged_ents`** 에 담으세요(리스트).

**확인 기준**: `bio_labels` 는 `['B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC']` 이고(`PER` 사람·`ORG` 기관·`LOC` 장소, 교안_01 4-1), `merged_ents` 는 **3개**입니다. 합친 쪽은 `entity` 대신 **`entity_group`** 칸을 쓰고 `start`·`end` 가 붙어 있어서, `news_sentence[start:end]` 가 `word` 와 똑같이 나옵니다.

<details><summary>힌트</summary>

```text
접근방법:
- pipeline 을 두 번 만든다. 하나는 그냥, 하나는 aggregation_strategy 를 준다.
- 파이프라인에 문자열을 넣으면 dict 의 리스트가 나온다. 필요한 칸만 꺼낸다.

세부구현:
1. 준비 셀에서 pipeline 을 이미 가져왔다.
2. ner = pipeline(...) 를 만들고 raw_tags = ner(news_sentence) 로 받는다.
3. bio_labels 는 raw_tags 를 돌며 t['entity'] 만 모은다.
4. merged = pipeline(..., aggregation_strategy='simple') 를 따로 만들고 merged_ents = merged(news_sentence) 로 받는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert bio_labels == ['B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC'],     "entity 값만 순서대로 모으세요(word 가 아니라 entity 칸입니다)"
assert 'El' in [tag['word'] for tag in raw_tags],     "raw_tags 는 합치기 전 결과여야 합니다(모델이 Elastic 을 El 과 ##astic 두 토큰으로 쪼갭니다)"
assert len(merged_ents) == 3, 'B- 로 시작하는 자리가 셋이니 합친 개체도 셋입니다'
assert [e['entity_group'] for e in merged_ents] == ['PER', 'ORG', 'LOC'],     "합친 쪽은 entity 가 아니라 entity_group 칸을 씁니다"
assert all(news_sentence[e['start']:e['end']] == e['word'] for e in merged_ents),     'start 와 end 로 원문을 잘라내면 word 와 같아야 합니다'
print('✅ 통과!')

---
# 2. 개체 스키마 정의

모델이 돌려줄 모양을 pydantic 으로 미리 정해 둡니다. 개체 하나짜리와 개체 목록짜리, 두 벌을 만듭니다(교안_02 2-1).

## 2-1. 개체 스키마 정의하기
**배경**: LLM 이 뽑은 개체를 프로그램이 다루려면 **정해진 구조**로 받아야 합니다. 개체 하나의 스키마를 만듭니다.

**요구사항**:
- `BaseModel` 을 상속한 **`Entity`** 클래스를 만드세요.
- 필드는 **`name`** 과 **`type`** 둘입니다. `name` 은 문자열이고, `type` 은 **`Literal['Compound', 'Gene', 'Disease', 'Symptom', 'PharmacologicClass', 'Other']`** 입니다(교안_02 2-1 과 같은 모양). 다섯 유형에 **`Other`** 를 더한 여섯입니다. 다섯에 안 맞는 개체를 모델이 억지로 뒤집어씌우지 않게 하는 칸입니다.
- 각 필드에 `Field(description=...)` 를 붙이세요. 이 문구는 주석이 아니라 **모델에게 그대로 전달되는 지시**라서 문구까지 채점합니다. **`'개체 이름(원문에 나온 그대로)'`** 과 **`'개체 유형'`** 으로 적으세요.

**예시**: `Entity(name='omeprazole', type='Compound')` 가 만들어지고 `.name` 이 `'omeprazole'` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- pydantic BaseModel 을 상속해 필드 두 개를 선언한다. type 은 str 이 아니라 Literal 이다.

세부구현:
1. BaseModel 과 Field 는 준비 셀에서 이미 가져왔다(다시 import 해도 된다).
2. class Entity(BaseModel): 아래에 name(str)과 type(Literal[여섯 값])을 선언한다. Literal 은 준비 셀에서 가져왔다.
3. 각 필드에 Field(description=...) 를 붙인다. 문구는 요구사항에 적힌 그대로 쓴다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert 'name' in Entity.model_fields and 'type' in Entity.model_fields, '필드 이름은 name 과 type 이어야 합니다'
assert Entity.model_fields['name'].description == '개체 이름(원문에 나온 그대로)', \
    "name 의 description 을 요구사항 문구('개체 이름(원문에 나온 그대로)')로 맞추세요"
assert Entity.model_fields['type'].description == '개체 유형', "type 의 description 을 '개체 유형' 으로 맞추세요"
sample = Entity(name='omeprazole', type='Compound')
assert sample.name == 'omeprazole' and sample.type == 'Compound', 'name 은 str, type 은 Literal 로 선언했는지 확인하세요'
# type 을 str 로 두면 아래가 통과해 버린다. Literal 이어야 목록 밖 값에서 막힌다.
try:
    Entity(name='CPIC', type='Guideline')
except Exception:
    pass
else:
    raise AssertionError('type 을 Literal 로 선언하세요. 지금은 목록 밖 유형도 그대로 들어갑니다')
print('✅ 통과!')

## 2-2. 개체 목록 스키마 정의하기
**배경**: 논문 한 편에는 개체가 여러 개입니다. 개체 **목록**을 담는 스키마를 만듭니다(2-1 과 같은 개념, 다른 모양).

**요구사항**:
- `BaseModel` 을 상속한 **`EntityList`** 클래스를 만드세요.
- 필드는 **`entities`** 하나이며, 타입은 **`list[Entity]`**(2-1 의 `Entity` 리스트)입니다.
- 여기엔 `Field(...)` 를 **붙이지 않아도 됩니다**. 필드가 하나뿐이고 이름이 `entities` 라 무엇을 담는지가 이미 분명하기 때문입니다(교안 3절은 설명을 붙였는데, 붙여도 틀린 것이 아닙니다).

**예시**: `EntityList(entities=[Entity(name='omeprazole', type='Compound')])` 의 `.entities` 길이는 1 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 2-1 의 Entity 를 원소로 갖는 리스트 필드 하나를 선언한다.

세부구현:
1. class EntityList(BaseModel): 를 만든다.
2. entities: list[Entity] 필드 하나만 선언한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert 'entities' in EntityList.model_fields, '필드 이름은 entities 하나여야 합니다'
# 원소 타입까지 본다. list 로만 두면 구조화 출력이 개체를 Entity 로 만들어 주지 않는다.
assert EntityList.model_fields['entities'].annotation == list[Entity],     'entities 의 타입을 list[Entity] 로 선언하세요(list 로만 두면 원소가 Entity 로 파싱되지 않습니다)'
box = EntityList(entities=[Entity(name='omeprazole', type='Compound')])
assert len(box.entities) == 1 and box.entities[0].name == 'omeprazole', \
    'entities 의 타입을 list[Entity] 로 선언했는지 확인하세요'
print('✅ 통과!')

---
# 3. 뽑은 개체 검증

뽑힌 개체를 그래프에 올려도 되는지 거릅니다. 같은 검사를 통과한 쪽과 걸린 쪽, 두 방향으로 봅니다(교안_02 3-1·3-2).

## 3-1. 유형 검증: 허용된 유형만 남기기
**배경**: 모델이 우리가 정하지 않은 유형을 붙이기도 합니다. 허용 유형(`allowed_types`)에 있는 개체만 남깁니다.

**요구사항**:
- `lv1_entities` 에서 `type` 이 `allowed_types` 에 **있는** 개체만 모아 **`valid_type`** 리스트에 담으세요.

**예시**: 허용 5종에 없는 유형(예: `'Procedure'`)은 빠집니다. 결과 개수는 20 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 리스트 컴프리헨션으로 type 이 allowed_types 에 있는 것만 고른다.

세부구현:
1. lv1_entities 를 돌며 각 개체의 type 을 검사한다.
2. allowed_types 에 있는 개체만 valid_type 에 모은다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(valid_type) == 20, 'lv1_entities 21개 중 허용 유형만 남기면 20개입니다'
assert all(e['type'] in allowed_types for e in valid_type), '허용되지 않은 유형이 섞여 있습니다'
assert all(e in lv1_entities for e in valid_type), '원본 lv1_entities 의 개체를 그대로 담아야 합니다'
print('✅ 통과!')

## 3-2. 원문 등장 확인: 지어낸 개체 걸러내기
**배경**: 모델이 원문에 없는 개체를 지어내기도 합니다(환각). 유형이 올바른 개체 중, 이름이 그 논문 원문에 **실제로 나오는** 것만 남깁니다.

**요구사항**:
- 아래 제공된 `valid_type`(유형이 올바른 20개) 중, 개체 이름이 `doc_text[개체의 doc_id]` 원문에 **들어 있는** 개체만 모아 **`present`** 리스트에 담으세요.

**예시**: 원문에 없는 개체가 빠져 결과는 18개입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 각 개체의 doc_id 로 원문을 찾아, 이름이 그 원문 문자열 안에 있는지 검사한다.

세부구현:
1. valid_type 를 돈다.
2. doc_text[e['doc_id']] 원문에 e['name'] 이 있는지(in) 검사한다.
3. 있는 개체만 present 에 모은다.
```

</details>

In [ ]:
# [제공 코드] 3-2, 3-3 의 입력입니다. 3-1 을 건너뛰었더라도 여기서부터 이어 풀 수 있게 다시 만듭니다.
valid_type = [e for e in lv1_entities if e['type'] in allowed_types]

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(present) == 18, 'valid_type 20개 중 원문에 없는 2개를 빼면 18개입니다'
assert all(e in valid_type for e in present), 'valid_type 의 개체를 그대로 담아야 합니다'
assert all(e['name'] in doc_text[e['doc_id']] for e in present), '그 개체의 doc_id 원문과 대조했는지 확인하세요'
print('✅ 통과!')

## 3-3. 환각 개체만 골라내기
**배경**: 반대로, 유형은 올바르지만 **원문에 없는**(지어낸) 개체의 이름만 모아 확인합니다(3-2 의 반대 방향).

**요구사항**:
- `valid_type` 중 이름이 원문에 **없는** 개체의 **이름만** 모아 **`hallucinated`** 리스트에 담으세요.
- 여기서 **원문**은 코퍼스 전체가 아니라 **그 개체가 나온 논문**입니다. 제공된 `doc_text[개체의 doc_id]` 와 대조하세요(코퍼스 전체와 대조하면 다른 논문에 우연히 있는 이름을 놓칩니다).

**예시**: 결과는 2개입니다(그럴듯하지만 그 논문이 말하지 않은 약물·유전자 이름).

<details><summary>힌트</summary>

```text
접근방법:
- 3-2 와 같되 조건을 not in 으로 뒤집고, 개체 전체가 아니라 name 만 모은다.

세부구현:
1. valid_type 를 돈다.
2. e['name'] 이 원문에 없는 개체의 e['name'] 만 모은다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(hallucinated) == 2, '유형은 맞지만 원문에 없는 개체는 2개입니다'
assert all(isinstance(n, str) for n in hallucinated), '개체 전체가 아니라 name 문자열만 담으세요'
assert set(hallucinated) <= {e['name'] for e in valid_type}, 'valid_type 에 있는 이름이어야 합니다'
assert hallucinated == [e['name'] for e in valid_type if e['name'] not in doc_text[e['doc_id']]], \
    "그 개체의 doc_id 원문과 대조해야 합니다(코퍼스 전체와 대조하면 다른 논문에 우연히 있는 이름을 놓칩니다)"
print('✅ 통과!')

---
# 4. 정리와 집계

거른 개체를 정리하고 세어 봅니다. 같은 개체가 두 번 든 것을 없애고, 유형별 분포를 냅니다(교안_02 3-3·4-1).

## 4-1. 중복 제거하기
**배경**: 같은 개체가 여러 논문에서, 또는 한 논문에서 두 번 뽑히기도 합니다. `(이름, 유형)` 이 같으면 하나만 남깁니다.

**요구사항**:
- `lv1_entities` 를 순회하며 `(name, type)` 쌍이 처음 나온 개체만 모아 **`deduped`** 리스트에 담으세요.
- 이미 본 쌍은 `set` 에 기록해 건너뜁니다.

**예시**: 중복이 빠져 결과는 20개입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 본 (이름, 유형) 을 집합에 기록하며, 처음 보는 것만 모은다.

세부구현:
1. 빈 집합 seen 과 빈 리스트 deduped 를 만든다.
2. 각 개체의 (name, type) 튜플을 키로 만든다.
3. 키가 seen 에 없으면 seen 에 넣고 deduped 에 추가한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(deduped) == 20, 'lv1_entities 21개에서 (이름, 유형) 중복 1개를 빼면 20개입니다'
# 이름만으로 묶으면 유형이 다른 개체까지 함께 사라진다. 그 오답을 여기서 가른다.
assert len(deduped) != len({e['name'] for e in lv1_entities}),     '이름만으로 묶으면 안 됩니다. 같은 이름이 다른 유형으로 뽑힌 개체는 서로 다른 개체입니다'
assert all(e in lv1_entities for e in deduped), '원본 lv1_entities 의 개체를 그대로 담아야 합니다'
keys = [(e['name'], e['type']) for e in deduped]
assert len(keys) == len(set(keys)), '같은 (이름, 유형) 이 두 번 들어 있습니다'
assert set(keys) == {(e['name'], e['type']) for e in lv1_entities}, \
    '중복만 빼야 합니다. 처음 보는 조합을 실수로 건너뛰지 않았는지 확인하세요'
assert deduped == [e for i, e in enumerate(lv1_entities)
                   if (e['name'], e['type']) not in {(x['name'], x['type']) for x in lv1_entities[:i]}], \
    '원본에 나온 순서 그대로, 처음 보는 (이름, 유형) 만 남겨야 합니다(정렬하지 마세요)'
print('✅ 통과!')

## 4-2. 유형별 개수 세기
**배경**: 뽑힌 개체가 어떤 유형에 몰려 있는지 보려면 유형별로 셉니다.

**요구사항**:
- `collections.Counter` 로 `lv1_entities` 전체의 **유형별 개수**를 세어 **`type_counts`** 에 담으세요.

**예시**: `type_counts['Gene']` 는 11 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 각 개체의 type 을 Counter 에 넣어 센다.

세부구현:
1. from collections import Counter
2. Counter 에 각 개체의 type 을 전달한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
from collections import Counter as _Counter

assert isinstance(type_counts, _Counter), \
    'collections.Counter 로 세어 담으세요(dict 를 손으로 적으면 세어 본 것이 아닙니다)'
assert type_counts['Gene'] == 11, 'Gene 유형 개수를 다시 세어 보세요'
assert sum(type_counts.values()) == len(lv1_entities), \
    '허용 5종만이 아니라 lv1_entities 전체를 세야 합니다(허용 밖 유형도 포함)'
assert type_counts == _Counter(e['type'] for e in lv1_entities), \
    'lv1_entities 를 Counter 로 센 값과 다릅니다(값을 손으로 적지 말고 세어 보세요)'
print('✅ 통과!')

---
# 5. LLM 으로 뽑기

앞에서 만든 스키마를 실제로 모델에 걸어 개체를 뽑고, 규칙 기반이 왜 여기서 막히는지 자기 말로 정리합니다(교안_02 2-1 · 교안_01 3-5).

## 5-1. 구조화 출력 추출기 만들기
**배경**: 1-1·1-2 에서 손으로 태깅한 그 문장을, 이번엔 **모델에게** 넘겨 봅니다. 2-1·2-2 에서 만든 **스키마**가 실제로 쓰이는 자리입니다.

**요구사항**:
- 원문 하나를 받아 개체 목록을 돌려주는 함수 **`extract_entities(text)`** 를 직접 작성하세요.
  - `make_model()` 에 **`with_structured_output`** 을 걸어 아래 제공된 **`EntityList`** 로 받게 합니다.
  - **`invoke`** 에는 메시지 두 개를 리스트로 넘깁니다. `role` 이 `system` 인 메시지에 제공된 **`EXTRACT_SYSTEM`**, `role` 이 `user` 인 메시지에 문장을 넣습니다.
  - 반환값의 **`.entities`** 를 돌려줍니다.
- 그 함수로 `sentence` 를 추출해 **`ents`** 에 담고, 각 개체의 `.name` 을 모아 **`names`** 리스트에 담으세요.

이 문제는 모델을 **1회** 부릅니다. 실제로 호출이 나가니 키가 있어야 풀 수 있습니다.

**예시**: `names` 에 `'gout'` 이 들어 있습니다(모델 출력이라 개수·표현은 조금 다를 수 있습니다).

<details><summary>힌트</summary>

```text
접근방법:
- 교안 3절에서 만든 추출기와 같은 모양이다. 모델에 스키마를 걸고, 메시지 두 개로 부른다.

세부구현:
1. 함수 안에서 make_model() 로 모델을 만들고 with_structured_output 에 EntityList 를 넘긴다.
2. invoke 에 딕셔너리 두 개가 든 리스트를 넘긴다.
   2-1. 첫 딕셔너리는 role 이 system, content 가 EXTRACT_SYSTEM.
   2-2. 둘째 딕셔너리는 role 이 user, content 가 함수가 받은 문장.
3. invoke 가 돌려준 값에서 entities 속성을 꺼내 돌려준다.
4. 함수를 sentence 로 부른 결과를 ents 에, 각 개체의 name 을 names 에 담는다.
```

</details>

In [ ]:
# [제공 코드] 5-1 입력: 스키마, system 문구, 문장입니다(실행만 하세요).
# 2-1, 2-2 에서 만든 것과 같은 스키마를 여기서 다시 정의합니다. 문제마다 앞 셀을 찾아 올라가지
# 않아도 되게, 이 문제에 필요한 것을 이 자리에 다 모아 둡니다.
class Entity(BaseModel):
    name: str = Field(description='개체 이름(원문에 나온 그대로)')
    type: Literal['Compound', 'Gene', 'Disease', 'Symptom', 'PharmacologicClass', 'Other'] = Field(description='개체 유형')
    other_type: Optional[str] = Field(
        default=None,
        description="type 이 Other 일 때만 채운다. 그 개체가 실제로 무엇인지 영어 한 낱말로 (예: Year, Institution)")
    confidence: float = Field(
        description="이 개체와 유형이 맞다고 얼마나 확신하는지 0.0 에서 1.0 사이")

class EntityList(BaseModel):
    entities: list[Entity]

EXTRACT_SYSTEM = '의학 논문 원문에서 개체를 뽑아라. 유형은 Compound, Gene, Disease, Symptom, PharmacologicClass 중에서 고르고, 다섯에 안 드는 개체는 Other 로 적는다. 개체 이름은 원문에 나온 표현을 그대로 쓴다.'
sentence = 'ATP is then metabolized into adenine and ultimately converted into uric acid, which elevates serum uric acid levels and increases the risk of gout.'
print('5-1 준비 완료:', sentence)

In [ ]:
# [제공 코드] 모델에 넘어간 요청을 기록해 둡니다: 이 셀은 실행만 하세요.
# 아래 문제들은 '모델을 실제로 부르는 것'이 요구 사항이라, 자가채점이 이 기록을 보고
# 답을 손으로 적어 넣지 않았는지 확인합니다.
from langchain_core.callbacks import BaseCallbackHandler

sent_prompts = []                    # 모델에 넘어간 요청 본문이 순서대로 쌓인다


class PromptRecorder(BaseCallbackHandler):
    """모델이 호출될 때마다 그 요청 본문을 sent_prompts 에 적어 둔다."""

    def on_chat_model_start(self, serialized, messages, **kwargs):
        # messages 는 [[메시지, 메시지, ...]] 꼴이다(한 번에 여러 대화를 보낼 수 있어서)
        for turn in messages:
            sent_prompts.append("\n".join(str(m.content) for m in turn))


recorder = PromptRecorder()


def make_model():
    # 준비 셀의 make_model 에 기록기를 달아 둔다. 이 뒤에 만드는 모델은 전부 여기를 지난다
    # 인자는 준비 셀과 똑같이 둡니다. 한쪽만 고치면 두 셀이 서로 다른 모델을 만듭니다
    return ChatOpenAI(model=MODEL_NAME, callbacks=[recorder])


print('요청 기록 준비 완료')

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert callable(extract_entities), 'extract_entities 라는 이름의 함수를 직접 만들어야 합니다'
assert len(ents) >= 2, '한 문장에서 개체를 여러 개 뽑아야 합니다'
assert all(isinstance(e, Entity) for e in ents), \
    'with_structured_output 에 EntityList 를 넘겨야 각 개체가 Entity 객체로 옵니다'
assert names == [e.name for e in ents], 'names 에는 각 개체의 .name 만 순서대로 담으세요'
assert any('gout' in n.lower() for n in names), '문장에 분명히 있는 개체가 빠졌습니다. system 문구를 그대로 넘겼는지 확인하세요'
assert any(sentence in prompt for prompt in sent_prompts),     '모델에 이 문장을 넘긴 기록이 없습니다. 개체를 손으로 적지 말고 extract_entities 로 실제 추출하세요'
print('✅ 통과!')

## 5-2. 규칙 기반의 한계 (서술형)
**배경**: 교안_01 의 규칙 기반 NER 은 표준 사전을 그대로 썼는데도 `statins` 를 못 잡았고, 형용사 `large` 를 유전자로 잡았습니다.

**요구사항**: 아래 markdown 셀에 다음을 **자신의 말로** 설명하세요. 정답 노트북의 모범 서술과 비교하세요.
- 규칙 기반 NER 이 **최근에 나온 약 이름**(사전에 없는 이름)을 못 잡는 이유
- 유전자 기호를 **소문자로 바꿔** 맞추면 안 되는 이유
- 반대로 **변이 id `rs9923231`** 같은 개체는 사전이 아니라 **패턴**으로 잡는 이유(교안 3-1)

*(여기에 자신의 설명을 서술하세요. 사전에 없는 이름을 못 잡는 이유 / 소문자로 바꿔 맞추면 안 되는 이유 / 변이 id 를 패턴으로 잡는 이유)*

---
수고했어요! LV1 에서 구간·BIO 태깅·개체 스키마·유형 검증·원문 등장 확인·중복 제거·집계·LLM 추출을 **하나씩** 익혔습니다. LV2 에서는 이것들을 **조합**해 정제 파이프라인과 규칙 대 LLM 비교를 다룹니다.